In [1]:
from pyspark.sql import SparkSession, functions as F
import sys; sys.executable
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point
from pyspark.sql.functions import col, when
import json

/Users/cherryxu/opt/anaconda3/lib/python3.9/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.1-CAPI-1.17.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/var/folders/l_/flqtngrd05d8qnbgmbtshrx00000gn/T/ipykernel_13856/892020361.py:4: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Assignment 2")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/09/13 08:22:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [25]:
gdf_bus_stop = gpd.read_file('../data/landing/PTV/PTV_METRO_BUS_STOP.shp')
gdf_train_station = gpd.read_file('../data/landing/PTV/PTV_METRO_TRAIN_STATION.shp')
gdf_tram_stop = gpd.read_file('../data/landing/PTV/PTV_METRO_TRAM_STOP.shp')
gdf_regional_train_station = gpd.read_file('../data/landing/PTV/PTV_REGIONAL_TRAIN_STATION.shp')

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/Users/cherryxu/opt/anaconda3/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Users/cherryxu/opt/anaconda3/lib/python3.9/socket.py", line 704, in readinto
    return self._sock.recv_into(b)
ConnectionResetError: [Errno 54] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/cherryxu/opt/anaconda3/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/cherryxu/opt/anaconda3/lib/python3.9/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [26]:
# Drop columns "STOP_ID" and "ROUTEUSSP" from the GeoDataFrame
gdf_train_station = gdf_train_station.drop(columns=["STOP_ID", "ROUTEUSSP"])
gdf_regional_train_station = gdf_regional_train_station.drop(columns=["STOP_ID"])
gdf_bus_stop = gdf_bus_stop.drop(columns=["STOP_ID", "ROUTEUSSP"])
gdf_tram_stop = gdf_tram_stop.drop(columns=["STOP_ID", "ROUTEUSSP"])

In [28]:
gdf_tram_stop

,LATITUDE,STOP_NAME,LONGITUDE,TICKETZONE,geometry
0,-37.744359,134-Merribell Ave/Nicholson St (Coburg),144.977728,1,POINT (2498037.678 2417399.046)
1,-37.811375,44-Deepdene Park/Whitehorse Rd (Balwyn),145.068671,1,POINT (2506047.181 2409959.283)
2,-37.811750,45-Hardwicke St/Whitehorse Rd (Balwyn),145.071785,1,POINT (2506321.345 2409917.457)
3,-37.812242,46-Balwyn Cinema/Whitehorse Rd (Balwyn),145.075930,1,POINT (2506686.248 2409862.575)
4,-37.812919,47-Balwyn Rd/Whitehorse Rd (Balwyn),145.081524,"1,2",POINT (2507178.746 2409787.031)
...,...,...,...,...,...
1660,-37.767614,34-Bent St/High St (Northcote),144.999096,1,POINT (2499920.889 2414818.327)
1661,-37.769413,33-Arthurton Rd/High St (Northcote),144.998900,1,POINT (2499903.606 2414618.673)
1662,-37.771138,32-Mitchell St/High St (Northcote),144.998569,1,POINT (2499874.461 2414427.220)
1663,-37.774712,31-Northcote Town Hall/High St (Northcote),144.997837,1,POINT (2499809.961 2414030.558)
